# Regularization

This notebook summarizes the key concepts from the DeepLearning.AI video on Regularization (L2 and L1) and weight decay.

## Overview

- Regularization is used to address overfitting (high variance).
- The main forms are L2 (Euclidean/weight decay) and L1 (sparsity).
- Regularization penalizes large weights, favoring simpler models.
- In neural networks, the penalty applies to all weight matrices `W^[l]` (not biases).
- `lambda` is the regularization hyperparameter (often written `lambd` in Python).

## Cost Function with Regularization

For logistic regression:

$$J(w,b) = \frac{1}{m} \sum_i \mathcal{L}(h_w(x^{(i)}), y^{(i)}) + \frac{\lambda}{2m} \|w\|_2^2$$

- L2 term: $$\frac{\lambda}{2m} \sum_j w_j^2$$
- L1 term: $$\frac{\lambda}{m} \sum_j |w_j|$$ (sometimes scaled by 2m for parity)

Bias `b` is usually not regularized (negligible effect vs high-dimensional `w`).

## L2 regularization (weight decay)

- In neural networks, J_reg adds $$\frac{\lambda}{2m} \sum_l \|W^{[l]}\|_F^2$$
- Gradient update for W^[l] becomes:
  - $$dW^{[l]} = dW^{[l]}_\text{orig} + \frac{\lambda}{m} W^{[l]}$$
  - $$W^{[l]} := W^{[l]} - \alpha dW^{[l]}$$
  - Equivalent to $$W^{[l]} := \left(1 - \frac{\alpha \lambda}{m}\right) W^{[l]} - \alpha dW^{[l]}_\text{orig}$$
- This multiplicative shrinkage is why L2 is called weight decay.

## L1 regularization (sparsity)

- Encourages many weights to be exactly zero.
- Can help with model compression and feature selection.
- Less commonly used for deep nets compared to L2.

## Lambda tuning strategy

- Use a dev/validation set or cross-validation.
- Try log-spaced lambda values (e.g., 0.0001, 0.001, 0.01, 0.1, 1).
- Choose the lambda that balances training accuracy and generalization.

In [1]:
# Practical lab: compare L2 and L1 regularization on a classification task
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# synthetic data with potential overfitting (more features than samples)
X, y = make_classification(n_samples=500, n_features=50, n_informative=8, n_redundant=10, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

results = []
for penalty in ['l2', 'l1']:
    for C in [100, 10, 1, 0.1, 0.01]:  # inverse regularization strength
        model = LogisticRegression(penalty=penalty, C=C, solver='saga', max_iter=5000, random_state=42)
        model.fit(X_train, y_train)
        train_acc = accuracy_score(y_train, model.predict(X_train))
        val_acc = accuracy_score(y_val, model.predict(X_val))
        weight_norm = np.linalg.norm(model.coef_)
        nnz = np.count_nonzero(model.coef_)
        results.append((penalty, C, train_acc, val_acc, weight_norm, nnz))

# show results in a concise table
print('penalty,C,train_acc,val_acc,||w||2,nnz')
for row in results:
    print(','.join(str(v) for v in row))

/Users/soubhik/AI/full-stack-ai-with-python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/soubhik/AI/full-stack-ai-with-python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/soubhik/AI/full-stack-ai-with-python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'pe

penalty,C,train_acc,val_acc,||w||2,nnz
l2,100,0.8028571428571428,0.74,1.3406487293487384,50
l2,10,0.8028571428571428,0.74,1.335921793152939,50
l2,1,0.8028571428571428,0.74,1.29122325946095,50
l2,0.1,0.8028571428571428,0.76,1.0195258643027443,50
l2,0.01,0.7971428571428572,0.7466666666666667,0.510905933899722,50
l1,100,0.8028571428571428,0.74,1.3390216323649808,50
l1,10,0.8028571428571428,0.74,1.3884893540562582,45
l1,1,0.8028571428571428,0.74,1.3936594113611716,35
l1,0.1,0.78,0.7266666666666667,0.7757823341488739,10
l1,0.01,0.6514285714285715,0.68,0.07087095098820927,1


/Users/soubhik/AI/full-stack-ai-with-python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/soubhik/AI/full-stack-ai-with-python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/soubhik/AI/full-stack-ai-with-python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 inst

## Interpretation of results

- As `C` decreases (stronger regularization), weight norms decrease and train accuracy may drop.
- Validation accuracy often peaks at moderate regularization (best generalization).
- L1 yields sparse weights (smaller `nnz`), L2 yields dense shrinkage.